# Deep Learning Lab Manual (CS 405)
## Experiment 10: Deep Reinforcement Learning for Agent-Environment Interaction

**Objective:** Implement a deep learning agent that learns to interact with and master the CartPole environment using Policy Gradient methods.

**Hardware:** Kaggle T4 x2 GPU  
**Time Required:** 3 hrs

## Step 0: GPU Setup & Verification (Kaggle T4 x2)

In [ ]:
import tensorflow as tf

# Configure both T4 GPUs for memory growth (prevents OOM errors)
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅ Found {len(gpus)} GPU(s):")
    for i, gpu in enumerate(gpus):
        print(f"   GPU {i}: {gpu.name}")
else:
    print("⚠️  No GPU found. Running on CPU.")

print(f"TensorFlow version: {tf.__version__}")

# Define MirroredStrategy for utilizing both T4 GPUs
# This distributes training across all available GPUs
strategy = tf.distribute.MirroredStrategy()
print(f"\n✅ MirroredStrategy initialized with {strategy.num_replicas_in_sync} replica(s)")

## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q gym==0.21.0
!pip install -q tensorflow_probability==0.14.0
!pip install -q pyglet==1.5.27
!pip install -q pyvirtualdisplay
!apt-get install -y -q xvfb x11-utils > /dev/null 2>&1
print("✅ Dependencies installed")

## Step 2: Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import gym
import time
import os
import io
import base64
from tqdm import tqdm
import tensorflow as tf
import tensorflow_probability as tfp
from IPython import display as ipythondisplay
from collections import deque

# Virtual display for rendering (Kaggle has no screen)
from pyvirtualdisplay import Display
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

print("✅ All imports successful")
print(f"Gym version: {gym.__version__}")
print(f"TF Probability version: {tfp.__version__}")

---
## Part 1: CartPole

### 3.1 Define the CartPole Environment and Agent

**CartPole:** Balance a pole, protruding from a cart, in an upright position by only moving the base left or right.

- **Observation Space (4 values):** Cart position, Cart velocity, Pole angle, Pole rotation rate
- **Action Space (2 actions):** Move LEFT (0) or Move RIGHT (1)
- **Reward:** +1 for every timestep the pole stays upright
- **Episode ends:** Pole > 15° from vertical OR cart moves > 2.4 units from center

In [ ]:
### Instantiate the CartPole environment ###
env = gym.make("CartPole-v1")
env.seed(1)

# Explore the observation space
n_observations = env.observation_space
print("Environment has observation space =", n_observations)

# Explore the action space
n_actions = env.action_space.n
print("Number of possible actions that the agent can choose from =", n_actions)

# Print observation space bounds for understanding
print("\nObservation space bounds:")
obs_labels = ["Cart Position", "Cart Velocity", "Pole Angle", "Pole Rotation Rate"]
for label, lo, hi in zip(obs_labels, env.observation_space.low, env.observation_space.high):
    print(f"  {label}: [{lo:.2f}, {hi:.2f}]")

### Define the CartPole Agent (Neural Network Policy)

A simple feed-forward neural network takes observations as input and outputs **action probabilities**.

In [ ]:
### Define the CartPole agent ###
# Using MirroredStrategy scope so the model is distributed across both T4 GPUs

def create_cartpole_model():
    """
    Feed-forward neural network for CartPole.
    Input:  4 observations (cart pos, cart vel, pole angle, pole rot rate)
    Output: 2 logits (one per action: left / right)
    """
    model = tf.keras.models.Sequential([
        # First Dense layer: 32 neurons, ReLU activation
        tf.keras.layers.Dense(units=32, activation='relu'),
        # TODO COMPLETED: Output layer — one logit per possible action (n_actions = 2)
        # No activation here; we treat these as raw logits for the categorical distribution
        tf.keras.layers.Dense(units=n_actions, activation=None)
    ])
    return model

# Build model inside strategy scope to utilize both GPUs
with strategy.scope():
    cartpole_model = create_cartpole_model()

# Build the model by passing a dummy input (required before summary)
dummy_obs = np.zeros((1, 4), dtype=np.float32)
_ = cartpole_model(dummy_obs)
cartpole_model.summary()
print("\n✅ CartPole model created inside MirroredStrategy scope (both GPUs)")

### Define the Agent's Action Function

This function:
1. Passes observations through the network to get **logits**
2. Samples an action from the resulting **categorical distribution**

In [ ]:
### Define the agent's action function ###
def choose_action(model, observation, single=True):
    """
    Forward pass through the model → sample an action.
    
    Args:
        model:       the neural network defining our agent
        observation: environment observation(s) as np.array
        single:      True  → single observation (shape [obs_dim])
                     False → batch of observations (shape [batch, obs_dim])
    Returns:
        action: integer action(s) sampled from the policy
    """
    # Add batch dimension for a single observation
    observation = np.expand_dims(observation, axis=0) if single else observation

    # TODO COMPLETED: Forward pass — get log-probabilities (logits) for each action
    logits = model.predict(observation, verbose=0)

    # TODO COMPLETED: Sample action from the categorical distribution defined by logits
    action = tf.random.categorical(logits, num_samples=1)  # shape: [batch, 1]

    action = action.numpy().flatten()  # convert tensor → numpy array
    return action[0] if single else action

# Quick sanity check
test_obs = env.reset()
test_action = choose_action(cartpole_model, test_obs)
print(f"Test observation shape: {test_obs.shape}")
print(f"Sample action for test observation: {test_action} ({'LEFT' if test_action == 0 else 'RIGHT'})")
print("✅ Action function working correctly")

---
## 3.2 Define the Agent's Memory

The `Memory` class stores observations, actions, and rewards collected during one **episode**.  
At the end of each episode, this data is used to update the policy.

In [ ]:
### Agent Memory ###
class Memory:
    def __init__(self):
        self.clear()

    def clear(self):
        """Reset/restart the memory buffer."""
        self.observations = []
        self.actions = []
        self.rewards = []

    def add_to_memory(self, new_observation, new_action, new_reward):
        """Add a single timestep's experience to memory."""
        self.observations.append(new_observation)
        # TODO COMPLETED: update the list of actions
        self.actions.append(new_action)
        # TODO COMPLETED: update the list of rewards
        self.rewards.append(new_reward)

    def __len__(self):
        return len(self.actions)

# Instantiate a single Memory buffer
memory = Memory()

# Quick test
memory.add_to_memory(np.zeros(4), 0, 1.0)
assert len(memory) == 1, "Memory add failed"
memory.clear()
assert len(memory) == 0, "Memory clear failed"
print("✅ Memory class working correctly")

---
## 3.3 Reward Function (Discounted Returns)

We compute the **discounted cumulative return** $R_t$ at each timestep:

$$R_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k}$$

- $\gamma \in (0,1)$ is the **discount factor** — rewards far in the future are worth less now
- We **normalize** the returns (zero mean, unit variance) to stabilize training

In [ ]:
### Reward function ###

def normalize(x):
    """Normalize array to zero mean and unit standard deviation."""
    x -= np.mean(x)
    x /= (np.std(x) + 1e-8)  # small epsilon to avoid division by zero
    return x.astype(np.float32)

def discount_rewards(rewards, gamma=0.95):
    """
    Compute normalized, discounted, cumulative rewards (returns).
    
    Args:
        rewards: list of rewards from an episode
        gamma:   discount factor (default 0.95)
    Returns:
        Normalized discounted returns as float32 array
    """
    discounted_rewards = np.zeros_like(rewards, dtype=np.float32)
    R = 0
    # Iterate backwards through the episode to accumulate discounted reward
    for t in reversed(range(len(rewards))):
        R = R * gamma + rewards[t]  # Bellman-style accumulation
        discounted_rewards[t] = R
    return normalize(discounted_rewards)

# Verify discount_rewards with a simple test
test_rewards = [1, 1, 1, 1, 1]  # 5 timesteps, each reward = 1
test_discounted = discount_rewards(test_rewards, gamma=0.95)
print("Test rewards:          ", test_rewards)
print("Discounted (raw):      ", [round(1 + 0.95 + 0.95**2 + 0.95**3 + 0.95**4, 4),
                                   round(1 + 0.95 + 0.95**2 + 0.95**3, 4),
                                   round(1 + 0.95 + 0.95**2, 4),
                                   round(1 + 0.95, 4), 1.0])
print("After normalize:       ", test_discounted)
print("✅ Discount rewards function working correctly")

---
## 3.4 Learning Algorithm — Policy Gradient

**Intuition:** Maximize the likelihood of actions that led to large rewards.

- **Loss:** Weighted negative log-likelihood of actions, scaled by their discounted returns
- **Formula:** $\mathcal{L} = -\mathbb{E}\left[\log \pi_\theta(a|s) \cdot R_t\right]$
- Good actions (high $R_t$) → their probability increases
- Bad actions (low $R_t$) → their probability decreases

In [ ]:
### Loss function ###
def compute_loss(logits, actions, rewards):
    """
    Policy gradient loss: negative log-prob of actions, weighted by returns.
    
    Args:
        logits:  network output — raw scores [batch, n_actions]
        actions: actions taken by the agent [batch]
        rewards: normalized discounted returns [batch]
    Returns:
        scalar loss value
    """
    # TODO COMPLETED: Compute negative log-probability of each taken action
    # sparse_softmax_cross_entropy_with_logits gives -log P(action | logits)
    neg_logprob = tf.nn.sparse_softmax_cross_entropy_with_logits(
        logits=logits,
        labels=actions
    )
    # TODO COMPLETED: Scale the negative log-probability by the discounted return
    # High return → amplify gradient; negative return → reverse gradient direction
    loss = tf.reduce_mean(neg_logprob * rewards)
    return loss

print("✅ Loss function defined")

In [ ]:
### Training step (forward pass + backpropagation) ###
def train_step(model, loss_function, optimizer, observations, actions, discounted_rewards, custom_fwd_fn=None):
    """
    One gradient update step.
    
    Args:
        model:               neural network policy
        loss_function:       compute_loss function
        optimizer:           e.g. Adam
        observations:        stacked observations from episode [T, obs_dim]
        actions:             actions taken in episode [T]
        discounted_rewards:  normalized discounted returns [T]
        custom_fwd_fn:       optional custom forward function
    """
    with tf.GradientTape() as tape:
        # Forward propagate through the agent network
        if custom_fwd_fn is not None:
            prediction = custom_fwd_fn(observations)
        else:
            prediction = model(observations)

        # TODO COMPLETED: Compute the policy gradient loss
        loss = loss_function(prediction, actions, discounted_rewards)

    # TODO COMPLETED: Backpropagation with gradient clipping
    # RL is noisy — gradient clipping prevents exploding gradients
    grads = tape.gradient(loss, model.trainable_variables)
    grads, _ = tf.clip_by_global_norm(grads, 2.0)  # clip norm at 2.0 (good for CartPole)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

print("✅ Training step defined")

---
## 3.5 Run CartPole Training!

The agent starts with **no knowledge** of the environment and learns purely from rewards.

**Expected behavior:** Rewards should increase steadily as training progresses, with the agent eventually balancing the pole for long episodes.

In [ ]:
## Training parameters ##
## Re-run this cell to restart training from scratch ##

# TODO COMPLETED: Learning rate and optimizer
learning_rate = 1e-3  # Adam with lr=0.001 works well for CartPole

# Build model and optimizer inside strategy scope → both T4 GPUs utilized
with strategy.scope():
    cartpole_model = create_cartpole_model()
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

# Build model by passing dummy input
_ = cartpole_model(np.zeros((1, 4), dtype=np.float32))

# Fresh memory and tracking containers
memory = Memory()
reward_history = []         # raw episode rewards
smoothed_rewards = []       # exponential moving average (α=0.05)
smoothing_factor = 0.95

print(f"Learning rate: {learning_rate}")
print(f"Optimizer: Adam")
print(f"Strategy replicas: {strategy.num_replicas_in_sync} GPU(s)")
print("\n✅ Ready to train! Run the next cell.")

In [ ]:
## CartPole Training Loop ##
# Note: stopping and restarting this cell picks up where you left off.
# To restart training, re-run the cell above first.

NUM_EPISODES = 500
PRINT_EVERY  = 50   # print progress every N episodes

print(f"Training for {NUM_EPISODES} episodes...\n")
start_time = time.time()

for i_episode in tqdm(range(NUM_EPISODES), desc="Training", unit="ep"):

    # ── Reset environment for a fresh episode ──────────────────────────────
    observation = env.reset()
    memory.clear()

    # ── Run one episode until termination ──────────────────────────────────
    while True:
        # Choose action using the current policy
        action = choose_action(cartpole_model, observation)

        # Step the environment
        next_observation, reward, done, info = env.step(action)

        # Store experience in memory
        memory.add_to_memory(observation, action, reward)

        if done:
            # ── Episode ended ───────────────────────────────────────────────
            total_reward = sum(memory.rewards)
            reward_history.append(total_reward)

            # Exponential moving average for smooth tracking
            if len(smoothed_rewards) == 0:
                smoothed_rewards.append(total_reward)
            else:
                smoothed_rewards.append(
                    smoothing_factor * smoothed_rewards[-1] + (1 - smoothing_factor) * total_reward
                )

            # ── Train on this episode's experience ──────────────────────────
            train_step(
                cartpole_model,
                compute_loss,
                optimizer,
                observations=np.vstack(memory.observations).astype(np.float32),
                actions=np.array(memory.actions, dtype=np.int32),
                discounted_rewards=discount_rewards(memory.rewards)
            )

            memory.clear()
            break  # next episode

        # Update observation for next step
        observation = next_observation

    # ── Periodic logging ────────────────────────────────────────────────────
    if (i_episode + 1) % PRINT_EVERY == 0:
        avg_last50 = np.mean(reward_history[-50:])
        print(f"  Episode {i_episode+1:4d} | "
              f"Last reward: {reward_history[-1]:6.1f} | "
              f"Avg last 50: {avg_last50:6.1f} | "
              f"Smoothed: {smoothed_rewards[-1]:6.1f}")

elapsed = time.time() - start_time
print(f"\n✅ Training complete in {elapsed:.1f}s ({elapsed/NUM_EPISODES:.2f}s/episode)")
print(f"   Best episode reward: {max(reward_history):.0f}")
print(f"   Final avg (last 50): {np.mean(reward_history[-50:]):.1f}")

---
## 3.6 Visualize Training Progress

In [ ]:
### Plot training rewards ###
plt.figure(figsize=(12, 5))

# Raw episode rewards
plt.subplot(1, 2, 1)
plt.plot(reward_history, alpha=0.4, color='steelblue', label='Episode Reward')
plt.plot(smoothed_rewards, color='navy', linewidth=2, label='Smoothed Reward (EMA)')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('CartPole Training — Policy Gradient')
plt.legend()
plt.grid(alpha=0.3)

# Rolling average (window=50)
plt.subplot(1, 2, 2)
window = 50
rolling_avg = [np.mean(reward_history[max(0, i-window):i+1]) for i in range(len(reward_history))]
plt.plot(rolling_avg, color='darkorange', linewidth=2, label=f'Rolling Avg (window={window})')
plt.axhline(y=475, color='green', linestyle='--', alpha=0.7, label='"Solved" threshold (475)')
plt.xlabel('Episode')
plt.ylabel('Average Reward')
plt.title(f'Rolling Average Reward (window={window})')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('cartpole_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Training curve saved as 'cartpole_training_curve.png'")

---
## 3.7 Evaluate the Trained Agent

In [ ]:
### Evaluate the trained CartPole agent over 20 test episodes ###
def evaluate_agent(model, env, n_episodes=20):
    """
    Run the trained agent for n_episodes and report performance.
    """
    eval_rewards = []
    for ep in range(n_episodes):
        obs = env.reset()
        total_reward = 0
        while True:
            action = choose_action(model, obs)
            obs, reward, done, _ = env.step(action)
            total_reward += reward
            if done:
                break
        eval_rewards.append(total_reward)

    print(f"Evaluation over {n_episodes} episodes:")
    print(f"  Mean reward : {np.mean(eval_rewards):.1f}")
    print(f"  Max reward  : {np.max(eval_rewards):.0f}")
    print(f"  Min reward  : {np.min(eval_rewards):.0f}")
    print(f"  Std reward  : {np.std(eval_rewards):.1f}")
    print(f"  Solved (≥475): {np.mean(np.array(eval_rewards) >= 475)*100:.0f}% of episodes")
    return eval_rewards

eval_env = gym.make("CartPole-v1")
eval_env.seed(42)
eval_rewards = evaluate_agent(cartpole_model, eval_env)

# Bar chart of evaluation rewards
plt.figure(figsize=(10, 4))
colors = ['green' if r >= 475 else 'steelblue' for r in eval_rewards]
plt.bar(range(len(eval_rewards)), eval_rewards, color=colors, edgecolor='white')
plt.axhline(y=475, color='red', linestyle='--', label='"Solved" (475)')
plt.axhline(y=np.mean(eval_rewards), color='orange', linestyle='-', label=f'Mean ({np.mean(eval_rewards):.1f})')
plt.xlabel('Evaluation Episode')
plt.ylabel('Total Reward')
plt.title('Trained Agent — Evaluation Rewards (green = solved)')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('cartpole_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3.8 Record & Display Agent Video

In [ ]:
### Record a video of the trained agent ###
import imageio

def record_agent_video(model, env_name="CartPole-v1", filename="cartpole_agent.gif", fps=30):
    """
    Record the trained agent and save as GIF.
    """
    rec_env = gym.make(env_name)
    rec_env.seed(0)
    obs = rec_env.reset()
    frames = []
    total_reward = 0

    while True:
        frame = rec_env.render(mode='rgb_array')
        frames.append(frame)
        action = choose_action(model, obs)
        obs, reward, done, _ = rec_env.step(action)
        total_reward += reward
        if done:
            break

    rec_env.close()
    imageio.mimsave(filename, frames, fps=fps)
    print(f"✅ Video saved as '{filename}' ({len(frames)} frames, total reward: {total_reward:.0f})")
    return filename

try:
    gif_path = record_agent_video(cartpole_model)
    # Display inline
    from IPython.display import Image
    display(Image(filename=gif_path, width=500))
except Exception as e:
    print(f"Video recording skipped (virtual display issue): {e}")
    print("The agent can still be evaluated numerically using evaluate_agent().")

---
## 3.9 Save the Trained Model

In [ ]:
### Save the trained model ###
SAVE_PATH = "cartpole_policy_model"
cartpole_model.save(SAVE_PATH)
print(f"✅ Model saved to '{SAVE_PATH}/'")

# Verify by loading
loaded_model = tf.keras.models.load_model(SAVE_PATH)
print("✅ Model loaded successfully — ready for inference")

---
## Exploration Tasks (Lab Report)

Answer the following questions based on your experiments:

In [ ]:
# ═══════════════════════════════════════════════════════════════
# EXPLORATION TASK 1: Effect of Learning Rate
# Try: learning_rate ∈ {1e-4, 1e-3, 5e-3, 1e-2}
# Question: How does learning rate affect stability and speed?
# ═══════════════════════════════════════════════════════════════

def run_experiment(lr, n_episodes=300, seed=42):
    """
    Run a CartPole experiment with a given learning rate.
    Returns the smoothed reward history.
    """
    exp_env = gym.make("CartPole-v1")
    exp_env.seed(seed)
    exp_memory = Memory()

    with strategy.scope():
        exp_model = create_cartpole_model()
        exp_optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    _ = exp_model(np.zeros((1, 4), dtype=np.float32))

    smoothed = []
    alpha = 0.05  # smoothing

    for ep in range(n_episodes):
        obs = exp_env.reset()
        exp_memory.clear()
        while True:
            action = choose_action(exp_model, obs)
            next_obs, reward, done, _ = exp_env.step(action)
            exp_memory.add_to_memory(obs, action, reward)
            if done:
                total_r = sum(exp_memory.rewards)
                smoothed.append(total_r if len(smoothed)==0 else
                                (1-alpha)*smoothed[-1] + alpha*total_r)
                train_step(exp_model, compute_loss, exp_optimizer,
                           np.vstack(exp_memory.observations).astype(np.float32),
                           np.array(exp_memory.actions, dtype=np.int32),
                           discount_rewards(exp_memory.rewards))
                exp_memory.clear()
                break
            obs = next_obs
    exp_env.close()
    return smoothed

# Experiment: compare learning rates
lrs = [1e-4, 1e-3, 5e-3, 1e-2]
plt.figure(figsize=(10, 5))
for lr in lrs:
    print(f"Running lr={lr}...")
    history = run_experiment(lr, n_episodes=300)
    plt.plot(history, label=f'lr={lr}')

plt.xlabel('Episode')
plt.ylabel('Smoothed Reward')
plt.title('Exploration Task 1: Effect of Learning Rate')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('lr_comparison.png', dpi=150)
plt.show()
print("\nObservation: Higher LR trains faster but may be unstable. Lower LR is stable but slow.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# EXPLORATION TASK 2: Effect of Discount Factor (gamma)
# Try: gamma ∈ {0.80, 0.90, 0.95, 0.99}
# Question: How does gamma affect the agent's time horizon?
# ═══════════════════════════════════════════════════════════════

def run_experiment_gamma(gamma, n_episodes=300, lr=1e-3, seed=42):
    exp_env = gym.make("CartPole-v1")
    exp_env.seed(seed)
    exp_memory = Memory()

    with strategy.scope():
        exp_model = create_cartpole_model()
        exp_optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    _ = exp_model(np.zeros((1, 4), dtype=np.float32))

    smoothed = []
    alpha = 0.05

    for ep in range(n_episodes):
        obs = exp_env.reset()
        exp_memory.clear()
        while True:
            action = choose_action(exp_model, obs)
            next_obs, reward, done, _ = exp_env.step(action)
            exp_memory.add_to_memory(obs, action, reward)
            if done:
                total_r = sum(exp_memory.rewards)
                smoothed.append(total_r if len(smoothed)==0 else
                                (1-alpha)*smoothed[-1] + alpha*total_r)
                train_step(exp_model, compute_loss, exp_optimizer,
                           np.vstack(exp_memory.observations).astype(np.float32),
                           np.array(exp_memory.actions, dtype=np.int32),
                           discount_rewards(exp_memory.rewards, gamma=gamma))
                exp_memory.clear()
                break
            obs = next_obs
    exp_env.close()
    return smoothed

gammas = [0.80, 0.90, 0.95, 0.99]
plt.figure(figsize=(10, 5))
for g in gammas:
    print(f"Running gamma={g}...")
    history = run_experiment_gamma(g, n_episodes=300)
    plt.plot(history, label=f'γ={g}')

plt.xlabel('Episode')
plt.ylabel('Smoothed Reward')
plt.title('Exploration Task 2: Effect of Discount Factor (γ)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('gamma_comparison.png', dpi=150)
plt.show()
print("\nObservation: γ=0.95–0.99 generally works best for CartPole (episode length ~200–500 steps).")

---
## Summary

| Component | Implementation |
|-----------|---------------|
| **Environment** | CartPole-v1 (OpenAI Gym) |
| **Agent** | 2-layer MLP: Dense(32, ReLU) → Dense(2, linear) |
| **Algorithm** | REINFORCE (Policy Gradient) |
| **Loss** | Weighted negative log-likelihood |
| **Reward shaping** | Discounted returns (γ=0.95), normalized |
| **Optimizer** | Adam (lr=1e-3) with gradient clipping (norm=2.0) |
| **GPU utilization** | TF MirroredStrategy — both Kaggle T4 GPUs |

**Key takeaways:**
- Policy gradient methods directly optimize the policy — no Q-table needed
- Discount factor γ balances short-term vs long-term reward optimization
- Gradient clipping is essential in RL due to high gradient variance
- MirroredStrategy distributes training across both GPUs, reducing wall-clock time